### 1. Imports and Configuration

In [8]:
import random
import pickle
from pathlib import Path

import numpy  as np
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from sympy             import nextprime, prime
from tqdm              import tqdm

# -----------------------------------------------------------------------
# Configuration -- mirrors notebook 00 exactly. Kept inline so this
# notebook runs stand-alone without requiring %run dependencies.
# -----------------------------------------------------------------------

VOCAB_SIZE   = 15
TOK_FIZZ     = 10
TOK_BUZZ     = 11
TOK_FZBZ     = 12
TOK_SEP      = 13
TOK_PAD      = 14
SEQ_LEN      = 128
N_SAMPLES    = 50000
TRAIN_FRAC   = 0.9
TASK_NAMES   = ['fibonacci', 'fizzbuzz', 'parity', 'primes']
SEED         = 42

ROOT_DIR     = Path('..').resolve()
DATA_DIR     = ROOT_DIR / 'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f'Vocab size   : {VOCAB_SIZE}')
print(f'Sequence len : {SEQ_LEN}')
print(f'Dataset size : {N_SAMPLES}  |  Tasks: {TASK_NAMES}')
# print(f'Data dir     : {DATA_DIR}')

Vocab size   : 15
Sequence len : 128
Dataset size : 50000  |  Tasks: ['fibonacci', 'fizzbuzz', 'parity', 'primes']


### 2. Vocabulary Helpers

In [2]:
# -----------------------------------------------------------------------
# Token codec -- integers to digit token lists and back.
# Numbers larger than 9 are encoded as multiple digit tokens.
# Example: 123 -> [1, 2, 3]
# -----------------------------------------------------------------------

TOKEN_TO_STR = {i: str(i) for i in range(10)}
TOKEN_TO_STR.update({TOK_FIZZ: 'F', TOK_BUZZ: 'B', TOK_FZBZ: 'FB', TOK_SEP: '|', TOK_PAD: '_'})


def int_to_tokens(n):
    '''
    Encode a non-negative integer as a list of single-digit tokens.

    Parameters
    ----------
    n : int   Non-negative integer to encode.

    Returns
    -------
    list[int]
        Sequence of digit tokens, most-significant digit first.
    '''
    return [int(d) for d in str(n)]


def tokens_to_str(tokens):
    '''
    Decode a token list to a human-readable string for inspection.

    Parameters
    ----------
    tokens : list[int]   Token indices.

    Returns
    -------
    str
        Space-separated string of token symbols.
    '''
    return ' '.join(TOKEN_TO_STR.get(t, '?') for t in tokens)


def fizzbuzz_tokens(n):
    '''
    Return the token(s) representing the FizzBuzz output for integer n.

    FizzBuzz rules:
      n % 15 == 0  ->  TOK_FZBZ
      n %  3 == 0  ->  TOK_FIZZ
      n %  5 == 0  ->  TOK_BUZZ
      else         ->  digit tokens for n

    Parameters
    ----------
    n : int   Positive integer.

    Returns
    -------
    list[int]
        One or more tokens representing the FizzBuzz output.
    '''
    if n % 15 == 0:
        return [TOK_FZBZ]
    elif n % 3 == 0:
        return [TOK_FIZZ]
    elif n % 5 == 0:
        return [TOK_BUZZ]
    else:
        return int_to_tokens(n)


# Quick sanity check on the codec
assert int_to_tokens(0)   == [0]
assert int_to_tokens(123) == [1, 2, 3]
assert fizzbuzz_tokens(3)  == [TOK_FIZZ]
assert fizzbuzz_tokens(5)  == [TOK_BUZZ]
assert fizzbuzz_tokens(15) == [TOK_FZBZ]
assert fizzbuzz_tokens(7)  == [7]

print('Vocabulary codec OK')
print(f'Sample: int_to_tokens(42)  -> {int_to_tokens(42)}')
print(f'Sample: fizzbuzz_tokens(15) -> {fizzbuzz_tokens(15)}  (TOK_FZBZ = {TOK_FZBZ})')
print(f'Token map: {TOKEN_TO_STR}')

Vocabulary codec OK
Sample: int_to_tokens(42)  -> [4, 2]
Sample: fizzbuzz_tokens(15) -> [12]  (TOK_FZBZ = 12)
Token map: {0: '0', 1: '1', 2: '2', 3: '3', 4: '4', 5: '5', 6: '6', 7: '7', 8: '8', 9: '9', 10: 'F', 11: 'B', 12: 'FB', 13: '|', 14: '_'}


### 3. Task Generators

In [3]:
# -----------------------------------------------------------------------
# Each generator produces a flat token list of length >= target_len.
# The caller truncates to the required length. Random starting points
# are used to prevent memorisation of a single sequence position.
# -----------------------------------------------------------------------

def gen_fibonacci(target_len):
    '''
    Generate a Fibonacci token stream with a random seed pair.

    Numbers are reduced modulo 1000 to cap digit length at 3, keeping
    sequences dense enough to contain many transitions per 128 tokens.
    Each number is followed by TOK_SEP.

    Parameters
    ----------
    target_len : int   Minimum number of tokens to produce.

    Returns
    -------
    list[int]
        Token stream of length >= target_len.
    '''
    a   = random.randint(0, 9)
    b   = random.randint(0, 9)
    out = []
    while len(out) < target_len:
        out.extend(int_to_tokens(a % 1000))
        out.append(TOK_SEP)
        a, b = b, (a + b) % 1000
    return out


def gen_fizzbuzz(target_len):
    '''
    Generate a FizzBuzz token stream starting from a random offset.

    Starting from a random integer in [1, 200] ensures the model sees
    diverse combinations of fizz/buzz/number patterns within each sample
    rather than always beginning at 1.

    Parameters
    ----------
    target_len : int   Minimum number of tokens to produce.

    Returns
    -------
    list[int]
        Token stream of length >= target_len.
    '''
    n   = random.randint(1, 200)
    out = []
    while len(out) < target_len:
        out.extend(fizzbuzz_tokens(n))
        out.append(TOK_SEP)
        n += 1
    return out


def gen_parity(target_len):
    '''
    Generate a parity task token stream.

    Each element of the stream is a bit string of random length [4, 14],
    followed by TOK_SEP, followed by the parity bit (0 or 1), followed
    by another TOK_SEP. The model must learn to count set bits across
    the entire bit string -- a known stress test for sequence integration.

    Parameters
    ----------
    target_len : int   Minimum number of tokens to produce.

    Returns
    -------
    list[int]
        Token stream of length >= target_len.
    '''
    out = []
    while len(out) < target_len:
        length = random.randint(4, 14)
        bits   = [random.randint(0, 1) for _ in range(length)]
        parity = sum(bits) % 2
        out.extend(bits)
        out.append(TOK_SEP)
        out.append(parity)
        out.append(TOK_SEP)
    return out


def gen_primes(target_len):
    '''
    Generate a consecutive-primes token stream from a random starting index.

    Primes are encoded digit-by-digit followed by TOK_SEP. The starting
    prime index (1-based, from sympy.prime) is drawn from [1, 100] so
    the model sees small and medium-sized primes but not astronomically
    large ones that would produce very long digit sequences.

    Parameters
    ----------
    target_len : int   Minimum number of tokens to produce.

    Returns
    -------
    list[int]
        Token stream of length >= target_len.
    '''
    start_idx = random.randint(1, 100)
    p         = prime(start_idx)
    out       = []
    while len(out) < target_len:
        out.extend(int_to_tokens(p))
        out.append(TOK_SEP)
        p = nextprime(p)
    return out


# Map task names to generator functions for use by the dataset class
TASK_GENERATORS = {
    'fibonacci' : gen_fibonacci,
    'fizzbuzz'  : gen_fizzbuzz,
    'parity'    : gen_parity,
    'primes'    : gen_primes,
}

# Smoke-test each generator -- verify token values are within vocab range
print('Generator smoke tests:')
for name, gen in TASK_GENERATORS.items():
    tokens = gen(SEQ_LEN + 1)
    assert all(0 <= t < VOCAB_SIZE for t in tokens), f'{name}: out-of-range token found'
    print(f'  {name:<12} len={len(tokens)}  first 20 tokens: {tokens_to_str(tokens[:20])}')

Generator smoke tests:
  fibonacci    len=129  first 20 tokens: 1 | 0 | 1 | 1 | 2 | 3 | 5 | 8 | 1 3 | 2
  fizzbuzz     len=132  first 20 tokens: B | 1 9 1 | F | 1 9 3 | 1 9 4 | FB | 1 9
  parity       len=135  first 20 tokens: 0 0 0 0 0 1 0 0 | 1 | 0 0 0 0 1 | 1 | 1
  primes       len=132  first 20 tokens: 2 7 7 | 2 8 1 | 2 8 3 | 2 9 3 | 3 0 7 |


### 4. Dataset Class

In [4]:
class SyntheticTaskDataset(Dataset):
    '''
    Multi-task synthetic sequence dataset for next-token prediction.

    Samples are drawn equally from all four tasks and shuffled so that
    no two consecutive batches are likely to contain the same task.
    No task label is exposed to the model -- the architecture must infer
    task identity from the sequence statistics alone.

    Each item returns:
      x          -- input tokens  [0 : SEQ_LEN]      shape (SEQ_LEN,)
      y          -- target tokens [1 : SEQ_LEN + 1]  shape (SEQ_LEN,)
      task_idx   -- integer task label for offline evaluation only

    Parameters
    ----------
    n_samples : int   Total samples across all tasks (must be divisible by 4).
    seq_len   : int   Fixed token sequence length.
    seed      : int   RNG seed for reproducibility.
    '''

    def __init__(self, n_samples, seq_len, seed=42):
        super().__init__()
        self.seq_len = seq_len
        per_task     = n_samples // len(TASK_NAMES)

        # Seed here so dataset contents are deterministic regardless of
        # what the caller has done to the global RNG before instantiation
        local_rng = random.Random(seed)

        samples  = []
        labels   = []

        for task_idx, name in enumerate(TASK_NAMES):
            gen = TASK_GENERATORS[name]
            for _ in range(per_task):
                # Generate seq_len + 1 tokens so input and target can
                # be formed by a simple one-position shift
                tokens = gen(seq_len + 1)

                # Truncate to exactly seq_len + 1; pad if unexpectedly short
                if len(tokens) < seq_len + 1:
                    tokens = tokens + [TOK_PAD] * (seq_len + 1 - len(tokens))
                tokens = tokens[: seq_len + 1]

                samples.append(tokens)
                labels.append(task_idx)

        # Shuffle with a seeded shuffle so the order is reproducible
        combined = list(zip(samples, labels))
        local_rng.shuffle(combined)
        samples, labels = zip(*combined)

        # Store as tensors up front to avoid repeated conversion during training
        self.samples = [torch.tensor(s, dtype=torch.long) for s in samples]
        self.labels  = list(labels)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        tokens = self.samples[idx]           # (seq_len + 1,)
        x      = tokens[:-1]                 # input:  positions 0 .. seq_len - 1
        y      = tokens[1:]                  # target: positions 1 .. seq_len
        return x, y, self.labels[idx]


print('SyntheticTaskDataset defined.')

SyntheticTaskDataset defined.


### 5. Dataset Generation and Inspection

In [ ]:
import time

# -----------------------------------------------------------------------
# Generate the full dataset, split into train / validation, and save
# to disk. Subsequent notebooks load from disk so generation only
# needs to run once.
# -----------------------------------------------------------------------

print('Generating dataset...')
t0      = time.perf_counter()
dataset = SyntheticTaskDataset(N_SAMPLES, SEQ_LEN, seed=SEED)
elapsed = time.perf_counter() - t0

print(f'Generated {len(dataset)} samples in {elapsed:.1f}s')

# Deterministic split using a fixed generator so the partition is stable
n_train = int(len(dataset) * TRAIN_FRAC)
n_val   = len(dataset) - n_train
train_ds, val_ds = random_split(
    dataset,
    [n_train, n_val],
    generator=torch.Generator().manual_seed(SEED),
)

print(f'Train samples   : {len(train_ds)}')
print(f'Val samples     : {len(val_ds)}')

# -----------------------------------------------------------------------
# Save the full dataset object and the split indices. Saving the full
# dataset (not the split subsets) lets future notebooks re-split with
# different fractions without regenerating everything.
# -----------------------------------------------------------------------

save_path = DATA_DIR / 'synthetic_dataset.pkl'
with open(save_path, 'wb') as f:
    pickle.dump({'dataset': dataset, 'train_indices': train_ds.indices, 'val_indices': val_ds.indices}, f)

# print(f'Saved to        : {save_path}')

In [6]:
# -----------------------------------------------------------------------
# Inspect a sample from each task to verify that the encoding is
# correct and that sequences look sensible before training.
# -----------------------------------------------------------------------

# Collect the first sample from each task by scanning the shuffled dataset
seen  = set()
shown = {}
for i in range(len(dataset)):
    _, _, task_idx = dataset[i]
    if task_idx not in seen:
        shown[task_idx] = i
        seen.add(task_idx)
    if len(seen) == len(TASK_NAMES):
        break

print('Sample inspection (first 40 tokens of input):')
print()

for task_idx, sample_idx in sorted(shown.items()):
    x, y, _ = dataset[sample_idx]
    name     = TASK_NAMES[task_idx]
    print(f'Task     : {name}  (idx {task_idx})')
    print(f'Input    : {tokens_to_str(x[:40].tolist())}')
    print(f'Target   : {tokens_to_str(y[:40].tolist())}')
    print(f'x shape  : {tuple(x.shape)}  |  unique tokens: {sorted(x.unique().tolist())}')
    print()

# -----------------------------------------------------------------------
# Per-task token distribution -- useful for diagnosing class imbalance
# that could distort cross-entropy loss comparisons between tasks.
# -----------------------------------------------------------------------

print('Token distribution per task (fraction of non-PAD tokens):')
print()

for task_idx, name in enumerate(TASK_NAMES):
    all_tokens = []
    for i in range(len(dataset)):
        x, _, tidx = dataset[i]
        if tidx == task_idx:
            all_tokens.extend(x[x != TOK_PAD].tolist())
        if len(all_tokens) > 50000:
            break    # Sample enough tokens without scanning the full dataset

    total  = len(all_tokens)
    counts = {t: all_tokens.count(t) for t in sorted(set(all_tokens))}
    top5   = sorted(counts.items(), key=lambda kv: -kv[1])[:5]
    print(f'  {name:<12}  total sampled: {total}')
    for tok, cnt in top5:
        sym = TOKEN_TO_STR.get(tok, str(tok))
        print(f'    token {tok} ({sym:<3})  {cnt / total * 100:.1f}%')
    print()

Sample inspection (first 40 tokens of input):

Task     : fibonacci  (idx 0)
Input    : 4 | 1 | 5 | 6 | 1 1 | 1 7 | 2 8 | 4 5 | 7 3 | 1 1 8 | 1 9 1 | 3 0 9 | 5 0 0 | 8
Target   : | 1 | 5 | 6 | 1 1 | 1 7 | 2 8 | 4 5 | 7 3 | 1 1 8 | 1 9 1 | 3 0 9 | 5 0 0 | 8 0
x shape  : (128,)  |  unique tokens: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 13]

Task     : fizzbuzz  (idx 1)
Input    : 5 2 | 5 3 | F | B | 5 6 | F | 5 8 | 5 9 | FB | 6 1 | 6 2 | F | 6 4 | B | F | 6 7
Target   : 2 | 5 3 | F | B | 5 6 | F | 5 8 | 5 9 | FB | 6 1 | 6 2 | F | 6 4 | B | F | 6 7 |
x shape  : (128,)  |  unique tokens: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]

Task     : parity  (idx 2)
Input    : 1 1 1 1 0 | 0 | 1 0 0 0 0 0 0 1 1 0 0 0 0 1 | 0 | 1 0 0 0 0 1 1 1 | 0 | 1 1 1 0
Target   : 1 1 1 0 | 0 | 1 0 0 0 0 0 0 1 1 0 0 0 0 1 | 0 | 1 0 0 0 0 1 1 1 | 0 | 1 1 1 0 1
x shape  : (128,)  |  unique tokens: [0, 1, 13]

Task     : primes  (idx 3)
Input    : 4 3 | 4 7 | 5 3 | 5 9 | 6 1 | 6 7 | 7 1 | 7 3 | 7 9 | 8 3 | 8 9 | 9 7 | 1 